# California Housing — Linear Regression and Regularisation

This notebook reproduces the lab: scale features, fit linear models, implement NumPy gradient descent, and compare Ridge / Lasso / ElasticNet.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Load data
data = fetch_california_housing(as_frame=True)
housing = data.frame
housing.shape

In [ ]:
# Look at the answer column
plt.hist(housing['MedHouseVal'], bins=50)
plt.title('Median House Value')
plt.show()

In [ ]:
# Split into training and test sets
X = housing.drop(columns='MedHouseVal')
y = housing['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
np.mean(X_train_s).round(3), np.std(X_train_s).round(3)

In [ ]:
# Fit with scikit-learn
lr = LinearRegression()
lr.fit(X_train_s, y_train)
pred = lr.predict(X_test_s)
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, pred)), 4))
print('R2:', round(r2_score(y_test, pred), 4))

In [ ]:
# Fit by hand using NumPy gradient descent
w = np.zeros(X_train_s.shape[1])
b = 0.0
rate = 0.1
losses = []
for step in range(500):
    pred_train = X_train_s @ w + b
    error = pred_train - y_train.to_numpy()
    losses.append((error ** 2).mean())
    w = w - rate * (X_train_s.T @ error) / len(y_train)
    b = b - rate * error.mean()
print('Final MSE:', round(losses[-1], 4))

In [ ]:
# Loss curve and coefficient comparison
plt.plot(losses)
plt.xlabel('step')
plt.ylabel('mean squared error')
plt.title('Gradient Descent Loss')
plt.show()
compare = pd.DataFrame({
    'feature': X.columns,
    'gradient_descent': np.round(w, 4),
    'sklearn': np.round(lr.coef_, 4),
})
compare

In [ ]:
# Ridge: test error against alpha
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_rmse = []
for a in alphas:
    model = Ridge(alpha=a)
    model.fit(X_train_s, y_train)
    ridge_rmse.append(np.sqrt(mean_squared_error(y_test, model.predict(X_test_s))))
plt.plot(alphas, ridge_rmse, marker='o')
plt.xscale('log')
plt.xlabel('alpha')
plt.ylabel('test RMSE')
plt.title('Ridge')
plt.show()

In [ ]:
# Lasso: test error against alpha
lasso_rmse = []
for a in alphas:
    model = Lasso(alpha=a, max_iter=10000)
    model.fit(X_train_s, y_train)
    lasso_rmse.append(np.sqrt(mean_squared_error(y_test, model.predict(X_test_s))))
plt.plot(alphas, lasso_rmse, marker='o')
plt.xscale('log')
plt.xlabel('alpha')
plt.ylabel('test RMSE')
plt.title('Lasso')
plt.show()

In [ ]:
# What the penalties did to the coefficients
coefs = pd.DataFrame({
    'LinearRegression': lr.coef_,
    'Ridge a=1': Ridge(alpha=1).fit(X_train_s, y_train).coef_,
    'Lasso a=0.1': Lasso(alpha=0.1, max_iter=10000).fit(X_train_s, y_train).coef_,
}, index=X.columns)
ax = coefs.plot.bar(figsize=(10, 4))
plt.ylabel('coefficient')
plt.title('Coefficients under each penalty')
plt.show()

In [ ]:
# Deliverable function and benchmark
def regression_benchmark(X_train, y_train, X_test, y_test, alphas):
    models = {'LinearRegression': LinearRegression()}
    for a in alphas:
        models[f'Ridge a={a}'] = Ridge(alpha=a)
        models[f'Lasso a={a}'] = Lasso(alpha=a, max_iter=10000)
        models[f'ElasticNet a={a}'] = ElasticNet(alpha=a, l1_ratio=0.5, max_iter=10000)
    rows = []
    for name in models:
        model = models[name]
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rows.append({
            'model': name,
            'test_rmse': np.sqrt(mean_squared_error(y_test, pred)),
            'test_r2': r2_score(y_test, pred),
            'zero_coefs': int((model.coef_ == 0).sum()),
        })
    table = pd.DataFrame(rows)
    table = table.sort_values('test_rmse').reset_index(drop=True)
    return table.round(4)

results = regression_benchmark(X_train_s, y_train, X_test_s, y_test, [0.01, 0.1, 1, 10])
results

**Next steps / How to run**:
- Create and activate a virtual environment and install dependencies from `requirements.txt`.
- Open this notebook in Jupyter or VS Code and run the cells.

Commands:
```powershell
python -m venv .venv
.\.venv\Scripts\activate
pip install -r requirements.txt
jupyter notebook
```